# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
%pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Note: you may need to restart the kernel to use updated packages.
Setup complete!


In [6]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [7]:
gold = pd.read_csv("./data/gold_standard_100.csv")
weak = pd.read_csv("./data/weak_labels_200.csv")
llm  = pd.read_csv("./data/llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence 
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label
X_llm = clf.predict(vec.transform(llm["review"])) # vectorize llm labels
preds = clf.predict_proba(vec.transform(llm["review"])) # predict probabaility for each class
confidence = np.max(preds, axis=1) # take max probability
mask = (confidence >= 0.65) & (X_llm == llm["label"]) # filter by confidence and label match
filtered_llm = llm[mask]
print(f"Filtered LLM: {len(filtered_llm)} / {len(llm)}")

#  1d. Merge & deduplicate 
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
merged = pd.concat([gold, weak, filtered_llm], ignore_index=True) # merge all datasets
merged = merged.drop_duplicates(subset="review").reset_index(drop=True) # drop duplicates
merged.to_csv("./data/consolidated_base.csv", index=False) # save to csv

#  1e. Class distribution analysis 
# TODO: Count per class, identify minority, plot and save class_distribution.png
counts = merged['label'].value_counts() # count number of classes
minority_class = counts.idxmin() # identify minority class
print("Minority class:", minority_class)

plt.figure(figsize=(6, 4)) # plot distribution
counts.plot(kind='bar', color=['blue', 'orange'])
plt.title('Class Distribution in Consolidated Dataset')
plt.xlabel('Sentiment Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("./data/class_distribution.png")

Gold: 100, Weak: 220, LLM: 150
Filtered LLM: 27 / 150
Minority class: Positive


In [8]:
#  1f. Augmentation functions
def get_wordnet_pos(tag):
     """Convert POS tag to WordNet format."""
     if tag.startswith('J'):
         return wordnet.ADJ
     elif tag.startswith('V'):
         return wordnet.VERB
     elif tag.startswith('N'):
         return wordnet.NOUN
     elif tag.startswith('R'):
         return wordnet.ADV
     else:
         return None
     
def get_synonyms(word, pos):
    """Get WordNet synonyms for a word with the given POS."""
    synonyms = set()
    for syn in wordnet.synsets(word, pos=pos):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ')
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    # TODO: Implement using nltk.corpus.wordnet, nltk.pos_tag, word_tokenize
    text = word_tokenize(text) # tokenize text
    pos_tags = pos_tag(text) # get POS tags # POS = part of speech represent grammatical role of a word in a sentence
    words_to_replace = [i for i, (word, pos) in enumerate(pos_tags) if word.lower() not in PRESERVE_WORDS and word.isalpha()] # identify words to replace
    n_replace = max(1, int(len(words_to_replace) * replace_frac)) # calculate number of words to replace
    indices = random.sample(words_to_replace, min(n_replace, len(words_to_replace))) # sample random tokens
    new_text = text.copy()
    replaced = 0
    for idx in indices:
        word, pos = pos_tags[idx]
        wn_pos = get_wordnet_pos(pos) # convert to WordNet POS
        if wn_pos:
            synonyms = get_synonyms(word, wn_pos) # find all synonyms
            
            if synonyms:                
                new_text[idx] = random.choice(synonyms) # replace with random synonym
                replaced += 1
                
            if replaced >= n_replace:
                break
    return " ".join(new_text)
    

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    # TODO: Implement with error handling and rate-limit sleep
    try:
        intm = GoogleTranslator(source=src, target=mid).translate(text) # translate to hindi
        back = GoogleTranslator(source=mid, target=src).translate(intm) # translate to english
        return back
    except Exception as e:
        print(f"Back-translation error: {e}")
        return text

def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    # TODO: Implement Jaccard similarity check
    og = set(original.lower().split()) # convert original to set of words
    aug = set(augmented.lower().split()) # convert augmented to set of words
    intersection = og.intersection(aug) # find intersection
    union = og.union(aug) # find union
    if not union:
        return False
    jaccard = len(intersection) / len(union) # jaccard = intersection / union
    return 0.30 <= jaccard <= 0.95 # filter by jaccard threshold

#  1g. Apply augmentation to minority class 
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
minority_samples = merged[merged['label'] == minority_class] # samples with minority class
augmented_data = []
for _, row in minority_samples.iterrows():
    review = row['review']
    label = row['label']
    
    # Synonym replacement
    aug1 = synonym_replacement(review) # generate synonym replacement
    if quality_filter(review, aug1): # filter by quality
        augmented_data.append({'review': aug1, 'label': label})
    
    # Back-translation
    aug2 = back_translate(review) # back translation augmentation
    if quality_filter(review, aug2): # filter by quality
        augmented_data.append({'review': aug2, 'label': label})
        
print(f"Generated {len(augmented_data)} augmented samples for minority class '{minority_class}'.")
# TODO: Save augmented_classical.csv
aug_df = pd.DataFrame(augmented_data)
aug_df.to_csv("./data/augmented_classical.csv", index=False) # save to csv

Generated 123 augmented samples for minority class 'Positive'.


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [9]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
review_examples = [] # sample 4 examples from gold standard for few-shot prompt
for i, (_, row) in enumerate(gold.sample(20, random_state=RANDOM_SEED).iterrows()):
    review_examples.append(f"{i+1}. Review: \"{row['review']}\" | Sentiment: {row['label']} | Movie: \"Example Movie\"")
    
# prompt for LLM
def generate_prompt(review_examples):
    return f"""You are an AI assistant that generates high-quality, realistic movie reviews.

Your task is to generate diverse movie reviews with clearly defined sentiments.

Each review must:
- Be natural and human-like (not robotic or repetitive)
- Mention a specific movie name (can be real or fictional)
- Clearly reflect ONE sentiment: Positive, Negative, or Neutral
- Be 2–5 sentences long
- Avoid repeating phrases or structures across reviews

{"\n".join(review_examples)}

Now generate exactly 20 NEW and UNIQUE reviews.

Sentiment distribution (strictly follow this):
- 10 Positive
- 6 Negative
- 4 Neutral

Output format:
Return ONLY a valid JSON array (no explanations, no extra text).

Example format:
[
    {{"review": "...", "sentiment": "Positive/Negative/Neutral", "movie": "..."}},
    {{"review": "...", "sentiment": "Positive/Negative/Neutral", "movie": "..."}},
]

Important rules:
- Do NOT repeat or copy the example reviews
- Do NOT include any text outside the JSON array
- Ensure diversity in wording, tone, and movie names
- Ensure the sentiment label matches the tone of the review exactly
"""

PROMPT_TEMPLATE = generate_prompt(review_examples[:4]) # generate prompt with 4 random examples

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors
generated_reviews = []
batch_size = 20
n_batches = 15
for i in range(n_batches):
    random.seed(RANDOM_SEED + i) # set seed for reproducibility
    PROMPT_TEMPLATE = generate_prompt(random.sample(review_examples, 4)) # generate new prompt with 4 random examples each batch
    print(f"Generating batch {i+1}/{n_batches}...")
    try:
        response = client.chat.completions.create( # request generation from LLM
            model=LLM_MODEL,
            messages=[{"role": "system", "content": PROMPT_TEMPLATE}],
            temperature=0.7,
            response_format={"type": "json_object"}
        )
        if response.choices and len(response.choices) > 0: # check if choices are returned
            content = response.choices[0].message.content
            if not content.strip():
                print(f"Empty response in batch {i+1}")
                continue
            try:
                content = content.replace("```json", "").replace("```", "").strip() # clean content 
                batch_reviews = json.loads(content) # parse JSON response
                if isinstance(batch_reviews, dict) and "reviews" in batch_reviews:
                    batch_reviews = batch_reviews["reviews"] # handle case where reviews are nested in a dict
                if isinstance(batch_reviews, list):
                    generated_reviews.extend(batch_reviews) # add to list if it's a list
                else:
                    print(f"Unexpected format in batch {i+1}: Not a list")
            except json.JSONDecodeError as e: # json decoding error
                print(f"JSON parsing error in batch {i+1}: {e}")
        else:            
            print(f"No choices returned in batch {i+1}")
    except Exception as e:
        print(f"Error in batch {i+1}: {e}")
    time.sleep(2) # delay for rate limiting
    
#print sentiment distribution of generated reviews
sentiment_counts = Counter([r.get("sentiment") for r in generated_reviews if "sentiment" in r]) # count sentiment distribution
print("Generated Sentiment Distribution:", sentiment_counts)

#  2c. Diversity metrics 
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score
def calculate_self_bleu(reviews):
    """Calculate average Self-BLEU for a list of reviews."""
    if len(reviews) < 2:
        return 0.0
    bleu_scores = []
    tokenized = [word_tokenize(r) for r in reviews] # tokenize reviews
    for i in range(len(reviews)): # calculate BLEU for each review against the rest
        reference = tokenized[:i] + tokenized[i+1:]
        candidate = tokenized[i]
        bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate)
        bleu_scores.append(bleu)
    return np.mean(bleu_scores) # return average BLEU score
sentiment_groups = {"Positive": [], "Negative": [], "Neutral": []}
for review in generated_reviews:
    sentiment = review.get("sentiment")
    if sentiment in sentiment_groups:
        sentiment_groups[sentiment].append(review.get("review", "")) # group reviews by sentiment

print("Sentiment Distribution:", {k: len(v) for k, v in sentiment_groups.items()})

diversity_report = {}
for sentiment, reviews in sentiment_groups.items():
    diversity_report[sentiment] = calculate_self_bleu(reviews) # calculate Self-BLEU for each sentiment group

for sentiment, score in diversity_report.items():
    if score >= 0.7:
        print(f"WARNING: Low diversity in {sentiment} (Self-BLEU={score:.3f})")


#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv
flagged_reviews = []
for review in generated_reviews:
    text = review.get("review", "")
    true_sentiment = review.get("sentiment", "")
    if text and true_sentiment:
        pred_sentiment = clf.predict(vec.transform([text]))[0] # predict sentiment using baseline model
        if pred_sentiment != true_sentiment: # flag if predicted sentiment doesn't match LLM label
            flagged_reviews.append({
                "review": text,
                "llm_sentiment": true_sentiment,
                "predicted_sentiment": pred_sentiment
            })
flagged_df = pd.DataFrame(flagged_reviews)
flagged_df.to_csv("./data/llm_generated_flagged.csv", index=False) # save to csv

#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt
with open("./data/diversity_report.txt", "w") as f:
    for sentiment, score in diversity_report.items():
        f.write(f"{sentiment}: {score:.4f}\n")
generated_df = pd.DataFrame(generated_reviews)
generated_df.to_csv("./data/llm_generated_300.csv", index=False) # save to csv

Generating batch 1/15...
Generating batch 2/15...
Generating batch 3/15...
Generating batch 4/15...
Generating batch 5/15...
Generating batch 6/15...
Generating batch 7/15...
Generating batch 8/15...
Generating batch 9/15...
Generating batch 10/15...
Generating batch 11/15...
Generating batch 12/15...
Generating batch 13/15...
Generating batch 14/15...
Generating batch 15/15...
Generated Sentiment Distribution: Counter({'Positive': 149, 'Negative': 90, 'Neutral': 61})
Sentiment Distribution: {'Positive': 149, 'Negative': 90, 'Neutral': 61}


## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [10]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)
consolidated = pd.read_csv("./data/consolidated_base.csv") # load consolidated dataset
consolidated['length'] = consolidated['review'].apply(lambda x: len(str(x).split()))
sampled_reviews = []
for label, count in [("Positive", 40), ("Negative", 40), ("Neutral", 20)]:
    subset = consolidated[consolidated['label'] == label]
    subset = subset.sample(frac=1, random_state=RANDOM_SEED) # shuffle subset
    subset = subset.sort_values('length').head(count) # sort by length and take shortest reviews for each class
    sampled_reviews.extend(subset.to_dict(orient='records')) # add to list of sampled reviews

#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits
translator = GoogleTranslator(source='en', target='hi')
back_translator = GoogleTranslator(source='hi', target='en')
for review in sampled_reviews:
    try:
        review['hindi'] = translator.translate(review['review']) # translate to hindi
    except Exception as e:
        review['hindi'] = "" # set hindi translation to empty string on error
        print(f"Error translating review: {e}")
    time.sleep(0.5)
print("Translation to Hindi completed for sampled reviews.")

#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"
smooth_fn = SmoothingFunction().method1
for review in sampled_reviews:
    try:
        if not review['hindi']:
            review['back_translated'] = "" # set back-translated text to empty string if hindi translation is missing
            review['bleu_score'] = 0.0 # set BLEU score to 0.0
            review['quality_flag'] = "FAIL" # set quality flag to FAIL if hindi translation is missing
            continue
        back = back_translator.translate(review['hindi']) # translate back to english
        review['back_translated'] = back
        reference = word_tokenize(review['review'].lower())
        candidate = word_tokenize(back.lower())
        bleu_score = sentence_bleu([reference], candidate, smoothing_function=smooth_fn) # calculate BLEU score
        review['bleu_score'] = bleu_score # store BLEU score
        review['quality_flag'] = "PASS" if bleu_score >= 0.3 else "FAIL" # store quality flag
    except Exception as e:
        review['back_translated'] = "" # set back-translated text to empty string on error
        review['bleu_score'] = 0.0 # set BLEU score to 0.0
        review['quality_flag'] = "FAIL" # set quality flag to FAIL on error
        print(f"Error in back-translation for review: {e}")
print("Back-translation and BLEU scoring completed for sampled reviews.")

#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label
for review in sampled_reviews:
    back_text = review['back_translated']
    original_label = review['label']
    if not back_text.strip():
        review['predicted_label'] = "Neutral" # default
        review['sentiment_preserved'] = ("Neutral" == original_label) # default
        continue
    pred_label = clf.predict(vec.transform([back_text]))[0]
    review['predicted_label'] = pred_label
    review['sentiment_preserved'] = (pred_label == original_label)

#  3e. Manual verification 
# TODO: Print 5 random samples for inspection
print("Sample augmented reviews:")
for review in random.sample(sampled_reviews, 5):
    print(f"Original: {review['review']}")
    print(f"Hindi: {review['hindi']}")
    print(f"Back-translated: {review['back_translated']}")
    print(f"Sentiment preserved: {review['sentiment_preserved']}")
    print()
    
with open("./data/manual_validation.txt", "w") as f:
    f.write("Observed Issues:\n")
    f.write("- Some back-translations are very close to the original, resulting in high BLEU but low diversity.\n")
    f.write("- Some times back-translations was nearly close to original, but sentiment was not preserved.\n")

#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag, sentiment_preserved
bilingual_df = pd.DataFrame(sampled_reviews)
bilingual_df = bilingual_df[["review", "label", "hindi", "back_translated", "bleu_score", "quality_flag", "sentiment_preserved"]] # select relevant columns
bilingual_df.to_csv("./data/bilingual_reviews.csv", index=False)

Translation to Hindi completed for sampled reviews.
Back-translation and BLEU scoring completed for sampled reviews.
Sample augmented reviews:
Original: I was completely blown away by this film. The opening scene was absolutely compelling. Don't miss this one.
Hindi: मैं इस फिल्म से पूरी तरह अभिभूत हो गया था। शुरूआती दृश्य बिल्कुल सम्मोहक था। इसे मत चूकिए.
Back-translated: I was completely enthralled by this film. The opening scene was absolutely mesmerizing. Don't miss it.
Sentiment preserved: True

Original: This is a very difficult movie to categorize. A polarizing experience for sure.
Hindi: इसे वर्गीकृत करना बहुत कठिन फिल्म है। निश्चित रूप से एक ध्रुवीकरण अनुभव।
Back-translated: This is a very difficult film to classify. Certainly a polarizing experience.
Sentiment preserved: True

Original: I have mixed feelings about this one. Good for a one-time watch.
Hindi: इसके बारे में मेरी मिश्रित भावनाएँ हैं। एक बार देखने के लिए अच्छा है।
Back-translated: I have mixed feelings about it. N

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [11]:
import string

from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class) 
# TODO: Sample from consolidated_base, mix short and long reviews
consolidated = pd.read_csv("./data/consolidated_base.csv")
consolidated['length'] = consolidated['review'].apply(lambda x: len(str(x).split()))
selected_reviews = []
for label, count in [("Positive", 10), ("Negative", 10), ("Neutral", 10)]:
    subset = consolidated[consolidated['label'] == label]
    subset = subset.sample(frac=1, random_state=RANDOM_SEED) # shuffle subset
    subset = subset.sort_values('length') # sort by length
    mix = pd.concat([subset.head(count//2), subset.tail(count//2)]) # take mix of short and long reviews
    selected_reviews.extend(mix.to_dict(orient='records')) # add to list of selected reviews

#  4b. TTS generation 
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile
for i, review in enumerate(selected_reviews):
    text = review['review']
    tts = gTTS(text=text, lang='en', tld='com') # generate TTS audio
    mp3_path = f"audio_samples/review_{i+1}.mp3" # mp3 path
    wav_path = f"audio_samples/review_{i+1}.wav" # wav path
    tts.save(mp3_path) # save as mp3
    
    y, sr = librosa.load(mp3_path, sr=16000)
    sf.write(wav_path, y, sr) # save as wav with soundfile

#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv
audio_features = []
for i in range(len(selected_reviews)):
    wav_path = f"audio_samples/review_{i+1}.wav"
    y, sr = librosa.load(wav_path, sr=16000) # load audio with librosa
    duration = librosa.get_duration(y=y, sr=sr) # calculate duration
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)) # calculate spectral centroid
    zero_crossing_rate = np.mean(librosa.feature.zero_crossing_rate(y)) # calculate zero crossing rate
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1) # calculate MFCCs
    
    features = {
        "review_id": i+1,
        "duration": duration,
        "spectral_centroid": spectral_centroid,
        "zero_crossing_rate": zero_crossing_rate,
    }
    for j in range(13):
        features[f"mfcc_{j+1}"] = mfccs[j]
    
    audio_features.append(features)
audio_features_df = pd.DataFrame(audio_features)
audio_features_df.to_csv("./data/audio_features.csv", index=False) # save to csv

#  4d. Whisper round-trip validation 
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv
def wer(reference, transcription): # calculate Word Error Rate
    """Calculate Word Error Rate between reference and transcription."""
    ref_words = reference.split()
    tcn_words = transcription.split()
    d = np.zeros((len(ref_words)+1, len(tcn_words)+1), dtype=np.uint8)
    for i in range(len(ref_words)+1):
        d[i][0] = i
    for j in range(len(tcn_words)+1):
        d[0][j] = j
    for i in range(1, len(ref_words)+1):
        for j in range(1, len(tcn_words)+1):
            if ref_words[i-1] == tcn_words[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                substitution = d[i-1][j-1] + 1
                insertion = d[i][j-1] + 1
                deletion = d[i-1][j] + 1
                d[i][j] = min(substitution, insertion, deletion)
    return d[len(ref_words)][len(tcn_words)] / max(len(ref_words), 1)

def clean_text(text):
    return text.translate(str.maketrans('', '', string.punctuation)).lower() # remove punctuation and lowercase

validation_results = []
for i, review in enumerate(selected_reviews):
    wav_path = f"audio_samples/review_{i+1}.wav"
    result = _whisper_model.transcribe(wav_path) # transcribe with Whisper
    transcription = clean_text(result['text'].strip()) # extract transcription
    original = clean_text(review['review'])
    error_rate = wer(original, transcription) # calculate WER
    validation_results.append({
        "review_id": i+1,
        "original": original,
        "transcription": transcription,
        "wer": error_rate,
        "flag": "FAIL" if error_rate > 0.25 else "PASS"
    })
validation_df = pd.DataFrame(validation_results)
validation_df.to_csv("./data/audio_validation.csv", index=False) # save to csv

## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [12]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv
consolidated_base = pd.read_csv("./data/consolidated_base.csv")
augmented_classical = pd.read_csv("./data/augmented_classical.csv")
llm_generated = pd.read_csv("./data/llm_generated_300.csv").rename(columns={"sentiment": "label"}) # rename sentiment column to label for consistency
bilingual_reviews = pd.read_csv("./data/bilingual_reviews.csv")
llm_flagged = pd.read_csv("./data/llm_generated_flagged.csv")
llm_flagged_reviews = set(llm_flagged['review']) # get set of flagged reviews to exclude
bilingual_reviews["review"] = bilingual_reviews['back_translated'] # use back-translated text for bilingual reviews
final_augmented = pd.concat([consolidated_base, augmented_classical, llm_generated, bilingual_reviews], ignore_index=True) # merge all datasets
final_augmented = final_augmented[~final_augmented['review'].isin(llm_flagged_reviews)] # exclude flagged LLM reviews
final_augmented = final_augmented.drop_duplicates(subset="review").reset_index(drop=True)
final_augmented = final_augmented[["review", "label"]] # keep only review and label columns
final_augmented.to_csv("./data/final_augmented_dataset.csv", index=False) # save to csv

#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("./data/gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)
baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)
augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
print(f"Baseline accuracy:  {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Initializing Black-Box Embedder...
Embedder loaded successfully.

--- Evaluating: Model ---
Training on 228 samples (excluded 100 test overlaps)...
Accuracy: 76.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.84      0.65        25
     Neutral       0.86      0.68      0.76        47
    Positive       1.00      0.82      0.90        28

    accuracy                           0.76       100
   macro avg       0.80      0.78      0.77       100
weighted avg       0.82      0.76      0.77       100

----------------------------------------

--- Evaluating: Model ---
Training on 577 samples (excluded 100 test overlaps)...
Accuracy: 83.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.62      0.96      0.75        25
     Neutral       0.97      0.74      0.84        47
    Positive       0.96      0.86      0.91        28

    accuracy                           0.83      